In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
primaryPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
scoredRows = predictions.join(evaluationRows.select('rowId', 'ndvi'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
metricExpressions = lambda: [pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), (1 - pl.col('error').pow(2).mean() / pl.col('ndvi').var(ddof=0)).alias('r2'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')]
foldMetrics = scoredRows.group_by('model', 'featureSet', 'seed', 'outerFold').agg(metricExpressions()).with_columns(pl.lit('fold').alias('aggregation')).rename({'outerFold': 'region'})
pooledMetrics = scoredRows.group_by('model', 'featureSet', 'seed').agg(metricExpressions()).with_columns(pl.lit('pooled').alias('aggregation'), pl.lit(None).cast(pl.Int64).alias('region'))
evaluationMetrics = pl.concat([foldMetrics, pooledMetrics], how='diagonal_relaxed').select('aggregation', 'region', 'model', 'featureSet', 'seed', 'mae', 'rmse', 'r2', 'bias', 'rowCount').sort('aggregation', 'region', 'model', 'featureSet', 'seed')
evaluationMetrics.write_parquet(primaryPath / 'evaluationMetrics.parquet')
assert evaluationMetrics.filter(pl.col('aggregation') == 'fold').height == 3 * 2 * 4 * 5
assert evaluationMetrics.filter(pl.col('aggregation') == 'pooled').height == 2 * 4 * 5
assert evaluationMetrics.select(pl.any_horizontal(pl.col('mae').is_null(), pl.col('rmse').is_null(), pl.col('r2').is_null(), pl.col('bias').is_null()).any()).item() is False
evaluationMetrics
